# Sample a city and build a Street View H5 store

Given a city name (or a custom boundary file), this notebook:

1. Loads the city boundary.
2. Downloads / loads the OSM drive road graph.
3. Samples points along road edges at a fixed metric spacing.
4. Runs the **free** Street View metadata endpoint to find which points have a panorama.
5. Downloads four-heading JPEGs per pano (this is the **paid** step).
6. Packs everything into a per-city H5 with the same schema as the existing Manchester store, plus extra fields (`pano_id`, `city`, `round_id`, `status`, `copyright`, `pano_lat/lon`, `sampled_lat/lon`).

**Resumability:** every stage is cached on disk. Interrupt and rerun any cell — the script picks up exactly where it stopped, and no API calls are repeated unnecessarily.

**Add more samples later:** rerun the sampling cell with a smaller `spacing_m` (denser) for the same city. New candidates are deduped against existing points (6dp lat/lon) and given fresh `point_id`s; existing data is untouched.

**Multiple cities:** each city gets its own folder under `data/<city_slug>/`. Use `merge_city_h5s` to combine them on demand.

**API key:** export `GOOGLE_STREETVIEW_API_KEY` before launching Jupyter.

In [ ]:
import importlib
import paths as paths_mod
import city_pipeline as cp
importlib.reload(paths_mod)
importlib.reload(cp)
from paths import CityPaths, get_api_key

## 1. Configure the run

- `CITY_NAME` is what OSM's geocoder will look up *unless* `POLYGON_PATH` is set, in which case the polygon file is used (and `CITY_NAME` is just used for the slug and for the H5 `city` field).
- `SPACING_M` controls how dense the sample is. 50 m gives a Manchester-scale dataset; try 200 m for a first pass on a new city.
- `MAX_POINTS` is a safety cap on how many images get downloaded in one go — useful for spot-checking on a new city before committing to a full run.

In [ ]:
CITY_NAME = "Leeds, UK"
POLYGON_PATH = None  # or e.g. "my_custom_boundary.geojson"
SPACING_M = 50.0
MAX_POINTS = None    # cap on points to send to image download; None = no cap

paths = CityPaths.for_city(CITY_NAME)
paths.ensure_dirs()
print("City slug:", paths.slug)
print("Data root:", paths.root)

## 2. Boundary and road graph

Both are cached. The first run pulls from OSM (slow); subsequent runs load from disk.

In [ ]:
boundary = cp.load_boundary(CITY_NAME, polygon_path=POLYGON_PATH)
graph = cp.build_road_graph(boundary, paths)
print(f"Graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges.")
boundary.plot(facecolor="none", edgecolor="black");

## 3. Sample points along edges (round 1, or N+1)

If `points.parquet` already has points for this city, only genuinely new candidates (6dp lat/lon not yet present) get appended. They are given fresh, non-colliding `point_id`s and tagged with a new `round_id`.

In [ ]:
candidates = cp.sample_points_along_edges(graph, spacing_m=SPACING_M)
print(f"{len(candidates)} candidate points before merging into master table.")
new_rows = cp.register_sampling_round(candidates, paths, spacing_m=SPACING_M)
new_rows.head()

In [ ]:
# Inspect the manifest of all sampling rounds done so far for this city.
import pandas as pd
pd.read_parquet(paths.samples_manifest)

## 4. Metadata pass (free)

This calls Google's Street View metadata endpoint, which is **free**. It tells us which points have a panorama, what `pano_id` they snap to, and what date the imagery is from. Pass `only_missing=False` to refresh.

In [ ]:
api_key = get_api_key()
meta = cp.fetch_metadata_for_points(paths, api_key, only_missing=True)
meta["status"].value_counts()

## 5. Image download (paid)

The first call below is a **dry run**: it prints how many image requests would be made and the estimated USD cost, but downloads nothing. Set `dry_run=False` in the next cell when you're ready to spend the money.

In [ ]:
_ = cp.download_images(paths, api_key, max_points=MAX_POINTS, dry_run=True)

In [ ]:
# Set dry_run=False to actually download. Safe to interrupt and rerun.
attempted = cp.download_images(paths, api_key, max_points=MAX_POINTS, dry_run=False)
attempted["status"].value_counts()

## 6. Build the H5

Reads the master point table + metadata + on-disk JPEGs and writes the per-city H5. Safe to rerun at any time — it's a full rebuild from disk state, so it always reflects whatever's been downloaded so far. Points that were attempted but have no image (no-pano or error) are kept as rows with `images_present=False`.

In [ ]:
h5_path = cp.build_h5(paths, city_name=CITY_NAME)
cp.show_h5_summary(h5_path)

## 7. Sanity check — render a row's images

In [ ]:
import io
import h5py
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

with h5py.File(h5_path, "r") as f:
    # Find the first row that has all four images present.
    present = f["images_present"][:]
    candidates = np.where(present.all(axis=1))[0]
    if len(candidates) == 0:
        candidates = np.where(present.any(axis=1))[0]
    row_idx = int(candidates[0])
    headings = list(np.asarray(f.attrs["headings"]))
    fig, axes = plt.subplots(1, len(headings), figsize=(4 * len(headings), 4))
    for j, ax in enumerate(axes):
        if not f["images_present"][row_idx, j]:
            ax.set_title(f"{headings[j]}\u00b0 (missing)")
            ax.axis("off")
            continue
        img = Image.open(io.BytesIO(f["images_jpeg"][row_idx, j].tobytes())).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{headings[j]}\u00b0")
        ax.axis("off")
    pid = int(f["point_id"][row_idx])
    plt.suptitle(f"point_id={pid}  pano={f['pano_id'][row_idx].decode('utf-8', 'ignore')}")
    plt.tight_layout()
    plt.show()

## 8. Optional: merge multiple cities into one combined H5

In [ ]:
# Example — uncomment to use.
# from pathlib import Path
# combined = cp.merge_city_h5s(
#     [CityPaths.for_city(n).h5 for n in ["Leeds, UK", "Sheffield, UK"]],
#     out_path=Path("data/combined_street_data.h5"),
# )